In [1]:
# Cell 1: Mount Drive and load files

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile

# Your Drive folder path
drive_path = '/content/drive/MyDrive/facebook-emotion-detector'

# Create local structure
os.makedirs('data/labeled', exist_ok=True)
os.makedirs('models/final_model', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Copy CSVs
for f in ['train.csv', 'val.csv', 'test.csv']:
    shutil.copy(f'{drive_path}/{f}', f'data/labeled/{f}')
    print(f"✅ {f} copied")

# Extract model
with zipfile.ZipFile(f'{drive_path}/final_model.zip', 'r') as z:
    z.extractall('models/final_model')
print("✅ Model extracted")

# Verify
print("\nFiles in model folder:")
print(os.listdir('models/final_model'))

Mounted at /content/drive
✅ train.csv copied
✅ val.csv copied
✅ test.csv copied
✅ Model extracted

Files in model folder:
['config.json', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'training_args.bin', 'README.md', 'model.safetensors']


In [2]:
# Cell 2: Install libraries
!pip install transformers datasets peft accelerate -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU: Tesla T4


In [3]:
# Cell 3: Reload model and tokenizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from datasets import load_from_disk
import pandas as pd
import numpy as np

# Load tokenizer and model from saved folder
tokenizer = AutoTokenizer.from_pretrained('models/final_model')
model = AutoModelForSequenceClassification.from_pretrained('models/final_model')

# Create pipeline for easy prediction
# device=0 means use GPU
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Load test data
test_df = pd.read_csv('data/labeled/test.csv')

print(f"✅ Model loaded!")
print(f"✅ Test set: {len(test_df)} comments")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights: 0it [00:00, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: models/final_model
Key                                                                                        | Status     | 
-------------------------------------------------------------------------------------------+------------+-
base_model.model.roberta.encoder.layer.{0...11}.attention.self.value.lora_A.default.weight | UNEXPECTED | 
base_model.model.roberta.encoder.layer.{0...11}.attention.self.value.lora_B.default.weight | UNEXPECTED | 
base_model.model.roberta.encoder.layer.{0...11}.attention.self.query.lora_B.default.weight | UNEXPECTED | 
base_model.model.roberta.encoder.layer.{0...11}.attention.self.query.lora_A.default.weight | UNEXPECTED | 
base_model.model.classifier.out_proj.bias                                                  | UNEXPECTED | 
base_model.model.classifier.dense.weight                                                   | UNEXPECTED | 
base_model.model.classifier.dense.bias                                 

✅ Model loaded!
✅ Test set: 126 comments


In [4]:
# Cell 4: Find Worst Errors
# Why: High confidence wrong predictions are most interesting
# because model was SURE but still failed — these reveal
# systematic weaknesses that the agent system must fix

import pandas as pd

test_df = pd.read_csv('data/labeled/test.csv')

print("Running predictions on test set...")
results = []

for idx, row in test_df.iterrows():
    pred = classifier(row['comment_text'])[0]

    # pred['label'] looks like 'LABEL_0', 'LABEL_1' etc
    # we need to map back to emotion names
    id2label = {0:'GP', 1:'GN', 2:'SAR', 3:'PA', 4:'ANG', 5:'CON', 6:'NEU'}
    pred_label = id2label[int(pred['label'].split('_')[1])]

    results.append({
        'comment':      row['comment_text'],
        'true_emotion': row['emotion'],
        'pred_emotion': pred_label,
        'confidence':   round(pred['score'], 3),
        'correct':      row['emotion'] == pred_label
    })

results_df = pd.DataFrame(results)

# Find errors only
errors_df = results_df[results_df['correct'] == False].copy()

# Sort by confidence descending
# High confidence + wrong = worst errors = most interesting
errors_df = errors_df.sort_values('confidence', ascending=False)

print(f"Total comments:  {len(test_df)}")
print(f"Correct:         {results_df['correct'].sum()}")
print(f"Errors:          {len(errors_df)}")
print(f"Error rate:      {len(errors_df)/len(test_df)*100:.1f}%")
print(f"\nTop 10 worst errors (high confidence but wrong):")
print(errors_df[['comment','true_emotion','pred_emotion','confidence']].head(10).to_string())

# Save all errors for manual review
errors_df.to_csv('outputs/top_50_errors.csv', index=False)
print("\n✅ Errors saved to outputs/top_50_errors.csv")

Running predictions on test set...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Total comments:  126
Correct:         30
Errors:          96
Error rate:      76.2%

Top 10 worst errors (high confidence but wrong):
                                                                                                                       comment true_emotion pred_emotion  confidence
68                                                                          গাজা খায় কিনা সেটা সিউর না বাট বাবা খায় এটা সিউর          SAR           GP       0.537
2                                                                                              Ai bedire dekhle aamr bumi ashe          ANG           GP       0.536
33                                                                                               তোমাকে দেখলেই নেশাখোর মনে হয়           GN           GP       0.536
16                                                                                                তুমি গাঁজা না খেলেও ছেলে খাস          SAR           GP       0.536
123  "The Prophet ﷺ said: He is not a bel